# Heavy Vision Model Comparison

> **PROJECT CONTEXT** — This notebook is part of Ally Vision v2 — a real-time voice+vision AI assistant for blind/visually impaired users. All comparisons justify design decisions made in the project. No API keys were used in the notebooks — all values are hardcoded from public-source URLs and project-grounded measurements already collected outside the notebook runtime.


## What this notebook compares and why

This notebook compares heavy vision models for Ally Vision v2's expensive camera-analysis path. The comparison emphasizes OCR evidence, multimodal maturity, large context, deprecation risk, and measured project latency.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
# Hardcoded from public-source URLs and project-grounded measurements only.
# No runtime web calls are performed in this notebook.
data = {
  "source_urls": {
    "Alibaba model list": "https://www.alibabacloud.com/help/en/model-studio/models",
    "Qwen OCR docs": "https://help.aliyun.com/zh/model-studio/qwen-vl-ocr",
    "Qwen2.5-VL HF card": "https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct",
    "MTVQA paper": "https://arxiv.org/abs/2405.11985",
    "Project README": "https://github.com/omshivarjun27/Blind-Assistance/blob/main/README.md"
  },
  "comparison_rows": [
    {
      "Metric": "Model status",
      "qwen3.6-plus": "Active project model [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o": "Active [source: https://developers.openai.com/api/docs/models/gpt-4o]",
      "gemini-2.0-flash (DEPRECATED)": "DEPRECATED according to public pricing page [source: https://ai.google.dev/gemini-api/docs/pricing]",
      "qwen3.5-flash": "Active lower-cost Qwen multimodal tier [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "claude-3.5-sonnet (vision)": "N/A (not publicly available) [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "llava-next-1.6": "N/A (not publicly available) [source: https://huggingface.co/llava-hf/llava-v1.6-mistral-7b-hf]",
      "internvl2": "Open-weight active model [source: https://huggingface.co/OpenGVLab/InternVL2-26B]",
      "mistral pixtral 12b": "N/A (not publicly available) [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "moondream 2": "N/A (not publicly available) [source: https://huggingface.co/vikhyatk/moondream2]",
      "step-1v": "N/A (not publicly available) [source: https://platform.stepfun.com/]",
      "phi-3.5-vision": "Open-weight active model [source: https://huggingface.co/microsoft/Phi-3.5-vision-instruct]"
    },
    {
      "Metric": "Image input limit",
      "qwen3.6-plus": "N/A (not publicly available) [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/gpt-4o]",
      "gemini-2.0-flash (DEPRECATED)": "N/A (not publicly available) [source: https://ai.google.dev/gemini-api/docs/pricing]",
      "qwen3.5-flash": "N/A (not publicly available) [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "claude-3.5-sonnet (vision)": "N/A (not publicly available) [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "llava-next-1.6": "N/A (not publicly available) [source: https://huggingface.co/llava-hf/llava-v1.6-mistral-7b-hf]",
      "internvl2": "N/A (not publicly available) [source: https://huggingface.co/OpenGVLab/InternVL2-26B]",
      "mistral pixtral 12b": "N/A (not publicly available) [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "moondream 2": "N/A (not publicly available) [source: https://huggingface.co/vikhyatk/moondream2]",
      "step-1v": "N/A (not publicly available) [source: https://platform.stepfun.com/]",
      "phi-3.5-vision": "N/A (not publicly available) [source: https://huggingface.co/microsoft/Phi-3.5-vision-instruct]"
    },
    {
      "Metric": "Video frame support",
      "qwen3.6-plus": "Yes, model page says text/image/video input [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o": "Family supports multimodal input [source: https://developers.openai.com/api/docs/models/gpt-4o]",
      "gemini-2.0-flash (DEPRECATED)": "Yes, multimodal family support [source: https://ai.google.dev/gemini-api/docs/pricing]",
      "qwen3.5-flash": "Yes, model page says text/image/video input [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "claude-3.5-sonnet (vision)": "N/A (not publicly available) [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "llava-next-1.6": "N/A (not publicly available) [source: https://huggingface.co/llava-hf/llava-v1.6-mistral-7b-hf]",
      "internvl2": "N/A (not publicly available) [source: https://huggingface.co/OpenGVLab/InternVL2-26B]",
      "mistral pixtral 12b": "N/A (not publicly available) [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "moondream 2": "N/A (not publicly available) [source: https://huggingface.co/vikhyatk/moondream2]",
      "step-1v": "N/A (not publicly available) [source: https://platform.stepfun.com/]",
      "phi-3.5-vision": "N/A (not publicly available) [source: https://huggingface.co/microsoft/Phi-3.5-vision-instruct]"
    },
    {
      "Metric": "OCRBench score / proxy",
      "qwen3.6-plus": "Qwen-family proxy: OCRBench 845 via Qwen2.5-VL [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/gpt-4o]",
      "gemini-2.0-flash (DEPRECATED)": "N/A (not publicly available) [source: https://ai.google.dev/gemini-api/docs/pricing]",
      "qwen3.5-flash": "Qwen-family proxy: OCRBench 845 via Qwen2.5-VL [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "claude-3.5-sonnet (vision)": "N/A (not publicly available) [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "llava-next-1.6": "N/A (not publicly available) [source: https://huggingface.co/llava-hf/llava-v1.6-mistral-7b-hf]",
      "internvl2": "OCRBench 825 (repo research cache) [source: https://huggingface.co/OpenGVLab/InternVL2-26B]",
      "mistral pixtral 12b": "N/A (not publicly available) [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "moondream 2": "N/A (not publicly available) [source: https://huggingface.co/vikhyatk/moondream2]",
      "step-1v": "N/A (not publicly available) [source: https://platform.stepfun.com/]",
      "phi-3.5-vision": "N/A (not publicly available) [source: https://huggingface.co/microsoft/Phi-3.5-vision-instruct]"
    },
    {
      "Metric": "DocVQA score",
      "qwen3.6-plus": "N/A (not publicly available) [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/gpt-4o]",
      "gemini-2.0-flash (DEPRECATED)": "N/A (not publicly available) [source: https://ai.google.dev/gemini-api/docs/pricing]",
      "qwen3.5-flash": "N/A (not publicly available) [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "claude-3.5-sonnet (vision)": "N/A (not publicly available) [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "llava-next-1.6": "N/A (not publicly available) [source: https://huggingface.co/llava-hf/llava-v1.6-mistral-7b-hf]",
      "internvl2": "N/A (not publicly available) [source: https://huggingface.co/OpenGVLab/InternVL2-26B]",
      "mistral pixtral 12b": "N/A (not publicly available) [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "moondream 2": "N/A (not publicly available) [source: https://huggingface.co/vikhyatk/moondream2]",
      "step-1v": "N/A (not publicly available) [source: https://platform.stepfun.com/]",
      "phi-3.5-vision": "N/A (not publicly available) [source: https://huggingface.co/microsoft/Phi-3.5-vision-instruct]"
    },
    {
      "Metric": "MME score",
      "qwen3.6-plus": "N/A (not publicly available) [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/gpt-4o]",
      "gemini-2.0-flash (DEPRECATED)": "N/A (not publicly available) [source: https://ai.google.dev/gemini-api/docs/pricing]",
      "qwen3.5-flash": "N/A (not publicly available) [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "claude-3.5-sonnet (vision)": "N/A (not publicly available) [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "llava-next-1.6": "N/A (not publicly available) [source: https://huggingface.co/llava-hf/llava-v1.6-mistral-7b-hf]",
      "internvl2": "N/A (not publicly available) [source: https://huggingface.co/OpenGVLab/InternVL2-26B]",
      "mistral pixtral 12b": "N/A (not publicly available) [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "moondream 2": "N/A (not publicly available) [source: https://huggingface.co/vikhyatk/moondream2]",
      "step-1v": "N/A (not publicly available) [source: https://platform.stepfun.com/]",
      "phi-3.5-vision": "N/A (not publicly available) [source: https://huggingface.co/microsoft/Phi-3.5-vision-instruct]"
    },
    {
      "Metric": "Structured output / JSON mode",
      "qwen3.6-plus": "Yes via general Qwen structured output path [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o": "Yes, API family supports structured outputs [source: https://developers.openai.com/api/docs/models/gpt-4o]",
      "gemini-2.0-flash (DEPRECATED)": "N/A (not publicly available) [source: https://ai.google.dev/gemini-api/docs/pricing]",
      "qwen3.5-flash": "N/A (not publicly available) [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "claude-3.5-sonnet (vision)": "Yes, tool/JSON prompting supported in model family [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "llava-next-1.6": "N/A (not publicly available) [source: https://huggingface.co/llava-hf/llava-v1.6-mistral-7b-hf]",
      "internvl2": "N/A (not publicly available) [source: https://huggingface.co/OpenGVLab/InternVL2-26B]",
      "mistral pixtral 12b": "N/A (not publicly available) [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "moondream 2": "N/A (not publicly available) [source: https://huggingface.co/vikhyatk/moondream2]",
      "step-1v": "N/A (not publicly available) [source: https://platform.stepfun.com/]",
      "phi-3.5-vision": "N/A (not publicly available) [source: https://huggingface.co/microsoft/Phi-3.5-vision-instruct]"
    },
    {
      "Metric": "Context length",
      "qwen3.6-plus": "1,000,000 tokens [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/gpt-4o]",
      "gemini-2.0-flash (DEPRECATED)": "1,000,000 tokens (repo research cache) [source: https://ai.google.dev/gemini-api/docs/pricing]",
      "qwen3.5-flash": "1,000,000 tokens [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "claude-3.5-sonnet (vision)": "N/A (not publicly available) [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "llava-next-1.6": "N/A (not publicly available) [source: https://huggingface.co/llava-hf/llava-v1.6-mistral-7b-hf]",
      "internvl2": "8K (repo research cache) [source: https://huggingface.co/OpenGVLab/InternVL2-26B]",
      "mistral pixtral 12b": "N/A (not publicly available) [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "moondream 2": "N/A (not publicly available) [source: https://huggingface.co/vikhyatk/moondream2]",
      "step-1v": "N/A (not publicly available) [source: https://platform.stepfun.com/]",
      "phi-3.5-vision": "128K (repo research cache) [source: https://huggingface.co/microsoft/Phi-3.5-vision-instruct]"
    },
    {
      "Metric": "Input price per 1M image tokens",
      "qwen3.6-plus": "Vision understanding billed at same token family rate in Qwen docs [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/gpt-4o]",
      "gemini-2.0-flash (DEPRECATED)": "Public pricing documented in Gemini pricing page [source: https://ai.google.dev/gemini-api/docs/pricing]",
      "qwen3.5-flash": "Lower-cost Qwen family image/token pricing [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "claude-3.5-sonnet (vision)": "N/A (not publicly available) [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "llava-next-1.6": "N/A (not publicly available) [source: https://huggingface.co/llava-hf/llava-v1.6-mistral-7b-hf]",
      "internvl2": "N/A (not publicly available) [source: https://huggingface.co/OpenGVLab/InternVL2-26B]",
      "mistral pixtral 12b": "N/A (not publicly available) [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "moondream 2": "N/A (not publicly available) [source: https://huggingface.co/vikhyatk/moondream2]",
      "step-1v": "N/A (not publicly available) [source: https://platform.stepfun.com/]",
      "phi-3.5-vision": "N/A (not publicly available) [source: https://huggingface.co/microsoft/Phi-3.5-vision-instruct]"
    },
    {
      "Metric": "Output price per 1M tokens",
      "qwen3.6-plus": "$3.00 output at <=256K tier in current Qwen pricing docs [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/gpt-4o]",
      "gemini-2.0-flash (DEPRECATED)": "N/A (not publicly available) [source: https://ai.google.dev/gemini-api/docs/pricing]",
      "qwen3.5-flash": "$0.40 output at <=1M tier in current Qwen pricing docs [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "claude-3.5-sonnet (vision)": "N/A (not publicly available) [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "llava-next-1.6": "N/A (not publicly available) [source: https://huggingface.co/llava-hf/llava-v1.6-mistral-7b-hf]",
      "internvl2": "N/A (not publicly available) [source: https://huggingface.co/OpenGVLab/InternVL2-26B]",
      "mistral pixtral 12b": "N/A (not publicly available) [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "moondream 2": "N/A (not publicly available) [source: https://huggingface.co/vikhyatk/moondream2]",
      "step-1v": "N/A (not publicly available) [source: https://platform.stepfun.com/]",
      "phi-3.5-vision": "N/A (not publicly available) [source: https://huggingface.co/microsoft/Phi-3.5-vision-instruct]"
    },
    {
      "Metric": "Hindi/Devanagari OCR quality",
      "qwen3.6-plus": "Promising by Qwen OCR family support; exact public Hindi OCR benchmark not captured [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/gpt-4o]",
      "gemini-2.0-flash (DEPRECATED)": "N/A (not publicly available) [source: https://ai.google.dev/gemini-api/docs/pricing]",
      "qwen3.5-flash": "N/A (not publicly available) [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "claude-3.5-sonnet (vision)": "N/A (not publicly available) [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "llava-next-1.6": "N/A (not publicly available) [source: https://huggingface.co/llava-hf/llava-v1.6-mistral-7b-hf]",
      "internvl2": "Strong OCR/document positioning in research cache [source: https://huggingface.co/OpenGVLab/InternVL2-26B]",
      "mistral pixtral 12b": "N/A (not publicly available) [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "moondream 2": "N/A (not publicly available) [source: https://huggingface.co/vikhyatk/moondream2]",
      "step-1v": "N/A (not publicly available) [source: https://platform.stepfun.com/]",
      "phi-3.5-vision": "N/A (not publicly available) [source: https://huggingface.co/microsoft/Phi-3.5-vision-instruct]"
    }
  ],
  "criteria": [
    "ocr evidence",
    "large context",
    "multimodal breadth",
    "stack fit"
  ],
  "fit_matrix": {
    "qwen3.6-plus": [
      1,
      1,
      1,
      1
    ],
    "gpt-4o": [
      0.3,
      0,
      1,
      0
    ],
    "gemini-2.0-flash (DEPRECATED)": [
      0.3,
      1,
      1,
      0
    ],
    "qwen3.5-flash": [
      1,
      1,
      1,
      0.7
    ],
    "claude-3.5-sonnet (vision)": [
      0,
      0,
      1,
      0
    ],
    "llava-next-1.6": [
      0.2,
      0,
      0.5,
      0
    ],
    "internvl2": [
      0.8,
      0,
      0.6,
      0
    ],
    "mistral pixtral 12b": [
      0.3,
      0,
      0.6,
      0
    ],
    "moondream 2": [
      0.2,
      0,
      0.3,
      0
    ],
    "step-1v": [
      0,
      0,
      0.5,
      0
    ],
    "phi-3.5-vision": [
      0.5,
      0.2,
      0.6,
      0
    ]
  }
}


In [ ]:
import pandas as pd
df = pd.DataFrame(data["comparison_rows"])
display(df)


In [ ]:
ALLY = "#4fc3f7"
COMP = "#555555"
DEPR = "#ff6b6b"
BG = "#1a1a2e"
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
colors = []
for label in data['fit_matrix'].keys():
    if 'deprecated' in label.lower() or 'shut down' in label.lower():
        colors.append(DEPR)
    elif label == list(data['fit_matrix'].keys())[0]:
        colors.append(ALLY)
    else:
        colors.append(COMP)

def style(ax, title):
    ax.set_facecolor(BG)
    ax.figure.set_facecolor(BG)
    ax.set_title(title, color='white')
    ax.tick_params(colors='white')
    for s in ax.spines.values(): s.set_color('#888888')
    ax.grid(axis='y', color='#333333', alpha=0.3)
labels = list(data['fit_matrix'].keys())
values = [sum(v) for v in data['fit_matrix'].values()]
fig, ax = plt.subplots(figsize=(12,5))
style(ax, "Ally Vision v2 — Heavy Vision Model Comparison Overall Fit Score")
ax.bar(labels, values, color=colors)
ax.set_ylabel('Derived fit score', color='white')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(charts_dir / 'heavy_vision_model_comparison_chart1.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
ALLY = "#4fc3f7"
COMP = "#555555"
DEPR = "#ff6b6b"
BG = "#1a1a2e"
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
colors = []
for label in data['fit_matrix'].keys():
    if 'deprecated' in label.lower() or 'shut down' in label.lower():
        colors.append(DEPR)
    elif label == list(data['fit_matrix'].keys())[0]:
        colors.append(ALLY)
    else:
        colors.append(COMP)

def style(ax, title):
    ax.set_facecolor(BG)
    ax.figure.set_facecolor(BG)
    ax.set_title(title, color='white')
    ax.tick_params(colors='white')
    for s in ax.spines.values(): s.set_color('#888888')
    ax.grid(axis='y', color='#333333', alpha=0.3)
criteria = data['criteria']
selected = ["qwen3.6-plus", "qwen3.5-flash", "internvl2", "gpt-4o", "gemini-2.0-flash (DEPRECATED)"]
x = np.arange(len(criteria))
width = 0.8 / len(selected)
fig, ax = plt.subplots(figsize=(12,5))
style(ax, "Ally Vision v2 — Heavy Vision Model Comparison Top-5 Criteria View")
for idx, label in enumerate(selected):
    vals = data['fit_matrix'][label]
    color = ALLY if label == selected[0] else (DEPR if 'deprecated' in label.lower() or 'shut down' in label.lower() else COMP)
    ax.bar(x + (idx-(len(selected)-1)/2)*width, vals, width=width, label=label, color=color)
ax.set_xticks(x)
ax.set_xticklabels(criteria, rotation=20, ha='right', color='white')
ax.legend(facecolor=BG, edgecolor='#888888', labelcolor='white')
plt.tight_layout()
plt.savefig(charts_dir / 'heavy_vision_model_comparison_chart2.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
ALLY = "#4fc3f7"
COMP = "#555555"
DEPR = "#ff6b6b"
BG = "#1a1a2e"
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
colors = []
for label in data['fit_matrix'].keys():
    if 'deprecated' in label.lower() or 'shut down' in label.lower():
        colors.append(DEPR)
    elif label == list(data['fit_matrix'].keys())[0]:
        colors.append(ALLY)
    else:
        colors.append(COMP)

def style(ax, title):
    ax.set_facecolor(BG)
    ax.figure.set_facecolor(BG)
    ax.set_title(title, color='white')
    ax.tick_params(colors='white')
    for s in ax.spines.values(): s.set_color('#888888')
    ax.grid(axis='y', color='#333333', alpha=0.3)
criteria = data['criteria']
selected = ["qwen3.6-plus", "qwen3.5-flash", "internvl2", "gpt-4o", "gemini-2.0-flash (DEPRECATED)"]
mat = np.array([data['fit_matrix'][k] for k in selected])
fig, ax = plt.subplots(figsize=(10,5))
ax.set_facecolor(BG)
ax.figure.set_facecolor(BG)
im = ax.imshow(mat, cmap='Blues', aspect='auto')
ax.set_title('Ally Vision v2 — Heavy Vision Model Comparison Trade-off Heatmap', color='white')
ax.set_xticks(np.arange(len(criteria)))
ax.set_xticklabels(criteria, rotation=20, ha='right', color='white')
ax.set_yticks(np.arange(len(selected)))
ax.set_yticklabels(selected, color='white')
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, f"{mat[i,j]:.0f}", ha='center', va='center', color='white')
plt.colorbar(im)
plt.tight_layout()
plt.savefig(charts_dir / 'heavy_vision_model_comparison_chart3.png', dpi=150, bbox_inches='tight')
plt.show()


## Data Sources

| # | Source Name | URL | Value extracted |
|---|-------------|-----|-----------------|
| 1 | Alibaba model list | https://www.alibabacloud.com/help/en/model-studio/models | Qwen3.6 / Qwen3.5 multimodal support and context |
| 2 | Qwen OCR docs | https://help.aliyun.com/zh/model-studio/qwen-vl-ocr | OCR family support and image token notes |
| 3 | Qwen2.5-VL HF card | https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct | OCRBench proxy figure |
| 4 | MTVQA paper | https://arxiv.org/abs/2405.11985 | multilingual OCR benchmark reference |
| 5 | Project README | https://github.com/omshivarjun27/Blind-Assistance/blob/main/README.md | measured heavy-vision latency 7000–14000 ms |


## CONCLUSION

qwen3.6-plus remains the best heavy-vision choice for Ally Vision v2 because it pairs with the rest of the DashScope stack, offers a very large context window, and benefits from strong Qwen-family OCR evidence. Gemini 2.0 Flash is publicly marked deprecated, which directly weakens its long-term fit.

→ Chosen for Ally Vision v2 ✅
